# Web Agent Action Prediction — Colab Pipeline

**최신화: 2026-05-06**

**전제**: Google Drive의 `DRIVE_ROOT` 아래에 준비:
- `my_code.zip` — `src/` 4개 파일을 zip 압축한 것
- `data/train.csv`, `data/test.csv`, `data/somenna_submission.csv`

**순서**: GPU 확인 → 설치 → 코드/데이터 → GPU 패치 → 학습 → 백업 → 추론 → 점검 → 저장

## 1. GPU 확인

In [ ]:
import subprocess
gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader']).decode().strip()
print('GPU:', gpu)
assert any(g in gpu for g in ['T4', 'A100', 'L4', 'V100']), f'지원하지 않는 GPU: {gpu}'
!nvidia-smi

GPU: NVIDIA L4
Wed May  6 14:41:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------------

## 2. 패키지 설치
런타임 재시작 불필요.

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "trl>=0.21" peft accelerate bitsandbytes
!pip install pandas tqdm scikit-learn

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-nalc8b1y/unsloth_c678c0d9344649df92a9ef450c01f9ca
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-nalc8b1y/unsloth_c678c0d9344649df92a9ef450c01f9ca
  Resolved https://github.com/unslothai/unsloth.git to commit fac2dc09b0fdf0c38a81f5bad889d58f6706d672
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.5.2-py3-none-any.whl size=32427785 sha256=e9c8a39085ca027034b8899e278e3ae4a98b2d577c7f8b13657c4887876ac697
  Stored in directory: /tmp/pip-ephem-wheel-cache-8k1sdfnv/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 35.7 MB/s eta 0:00:00
   ━━

## 3. Drive 마운트 + 코드/데이터 배치

`DRIVE_ROOT` 아래에 다음 파일이 있어야 합니다:
- `my_code.zip` — `src/` 폴더 안의 4개 파일(`preprocess.py`, `retrieval.py`, `train.py`, `inference.py`)을 zip 압축
- `data/train.csv`, `data/test.csv`, `data/somenna_submission.csv`

In [ ]:
from google.colab import files
import os, zipfile

os.makedirs('/content/src', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)

uploaded = files.upload()  # my_code.zip + 3개 csv 선택

# zip 압축 해제
with zipfile.ZipFile('my_code.zip') as z:
    z.extractall('/content/src/')

# csv 이동
import shutil
for f in ['train.csv', 'test.csv', 'somenna_submission.csv']:
    if os.path.exists(f):
        shutil.move(f, f'/content/data/{f}')

!ls /content/src && echo '---' && ls /content/data

Saving my_code.zip to my_code (1).zip
inference.py  preprocess.py  retrieval.py  train.py
---
somenna_submission.csv	test.csv  train.csv


## 4. GPU별 설정 자동 패치

| GPU | VRAM | batch | grad_accum | 유효배치 | max_steps | 학습시간 | inf batch |
|-----|------|-------|------------|---------|-----------|---------|----------|
| **A100** | 40GB | 32 | 1 | 32 | 2400 | ~2.1h | 32 |
| L4  | 24GB | 2 | 8 | 16 | 1200 | ~4.3h | 8 |
| T4  | 16GB | 1 | 16 | 16 | 800  | ~4.0h | 4 |

> **A100 권장**: 3시간 세션 안에 학습+추론 완결 가능


In [ ]:
import subprocess, re

gpu_name = subprocess.check_output(
    ["nvidia-smi","--query-gpu=name","--format=csv,noheader"]
).decode().strip()

is_a100 = "A100" in gpu_name
is_l4   = "L4"   in gpu_name
profile = "A100" if is_a100 else ("L4" if is_l4 else "T4")
print(f"GPU: {gpu_name} → {profile} 프로파일 적용")

tp = "/content/src/train.py"
ip = "/content/src/inference.py"

if is_a100:
    s = open(tp).read()
    s = re.sub(r"per_device_train_batch_size\s*=\s*\d+", "per_device_train_batch_size = 32", s)
    s = re.sub(r"gradient_accumulation_steps\s*=\s*\d+", "gradient_accumulation_steps = 1", s)
    s = re.sub(r"max_steps\s*=\s*\d+", "max_steps = 2400", s)
    s = re.sub(r"dataset_num_proc\s*=\s*\d+", "dataset_num_proc = 4", s)
    open(tp, "w").write(s)
    s = open(ip).read()
    s = re.sub(r"^BATCH_SIZE\s*=\s*\d+", "BATCH_SIZE = 32", s, flags=re.M)
    open(ip, "w").write(s)
    print("A100 패치 완료: train(batch=32, grad=1, steps=2400) / inference(batch=32)")

elif is_l4:
    s = open(tp).read()
    s = re.sub(r"per_device_train_batch_size\s*=\s*\d+", "per_device_train_batch_size = 2", s)
    s = re.sub(r"gradient_accumulation_steps\s*=\s*\d+", "gradient_accumulation_steps = 8", s)
    s = re.sub(r"max_steps\s*=\s*\d+", "max_steps = 1200", s)
    s = re.sub(r"dataset_num_proc\s*=\s*\d+", "dataset_num_proc = 4", s)
    open(tp, "w").write(s)
    s = open(ip).read()
    s = re.sub(r"^BATCH_SIZE\s*=\s*\d+", "BATCH_SIZE = 8", s, flags=re.M)
    open(ip, "w").write(s)
    print("L4 패치 완료: train(batch=2, grad=8, steps=1200) / inference(batch=8)")

else:  # T4
    s = open(tp).read()
    s = re.sub(r"per_device_train_batch_size\s*=\s*\d+", "per_device_train_batch_size = 1", s)
    s = re.sub(r"gradient_accumulation_steps\s*=\s*\d+", "gradient_accumulation_steps = 16", s)
    s = re.sub(r"max_steps\s*=\s*\d+", "max_steps = 800", s)
    open(tp, "w").write(s)
    s = open(ip).read()
    s = re.sub(r"^BATCH_SIZE\s*=\s*\d+", "BATCH_SIZE = 4", s, flags=re.M)
    open(ip, "w").write(s)
    print("T4 패치 완료: train(batch=1, grad=16, steps=800) / inference(batch=4)")


GPU: NVIDIA L4 → L4 프로파일 적용
L4 패치 완료: train(4,4) / inference(8)


## 5. 학습 (LoRA SFT)

- 모델: `Qwen2.5-3B-Instruct` (4bit)
- LoRA: r=16, 7개 projection
- max_steps: 3000, cosine scheduler
- 학습 제외: `site_2aa627db`

산출물: `/content/lora_model/`, `/content/outputs/eval_metrics.json`

In [ ]:
%cd /content
!python src/train.py

/content
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: NVIDIA L4 | VRAM: 22.0 GB
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.0.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
model.safetensors: 100% 2.05G/2.05G [00:07<00:00, 277MB/s]
Loading weights: 100% 434/434 [00:00<00:00, 583.36it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 271/271 [00:00<00:00, 1.60MB/s]
config.json: 1.34kB [00:00, 4.66MB/s]
tokenizer_config.json: 7.36kB [00:00, 11.8MB/s]
vocab.json: 2.78MB [00:00, 146MB/s]
merges.txt: 1.67MB [00:00, 110MB/s]
t

## 6. 학습 결과 확인

> **[Claude 공유 필수 #1]**  
> 아래 셀을 실행한 뒤 **출력 결과 전체를 복사해서 Claude에게 붙여넣기**  
> (workflow / real_web 각각의 op_acc / target_acc / value_acc / exact_match 값이 핵심)


In [ ]:
import json
with open('/content/outputs/eval_metrics.json') as f:
    metrics = json.load(f)

print('=== 검증 결과 ===')
for key, m in metrics.items():
    if m.get('n', 0) == 0:
        continue
    print(f"\n[{key}] n={m['n']}")
    print(f"  op_acc      : {m['op_acc']:.3f}")
    print(f"  target_acc  : {m['target_id_acc']:.3f}")
    print(f"  value_acc   : {m['value_acc']:.3f}")
    print(f"  exact_match : {m['exact_match']:.3f}  ← 대회 기준")

=== 검증 결과 ===

[overall] n=300
  op_acc      : 0.953
  target_acc  : 0.670
  value_acc   : 0.920
  exact_match : 0.630  ← 대회 기준

[site_unseen] n=300
  op_acc      : 0.953
  target_acc  : 0.670
  value_acc   : 0.920
  exact_match : 0.630  ← 대회 기준

[workflow] n=126
  op_acc      : 1.000
  target_acc  : 1.000
  value_acc   : 1.000
  exact_match : 1.000  ← 대회 기준

[real_web] n=174
  op_acc      : 0.920
  target_acc  : 0.431
  value_acc   : 0.862
  exact_match : 0.362  ← 대회 기준


## 7. lora_model Drive 백업

In [ ]:
import os
!mkdir -p "$DRIVE_ROOT/lora_model"
!cp -r /content/lora_model/* "$DRIVE_ROOT/lora_model/"
!cp /content/outputs/eval_metrics.json "$DRIVE_ROOT/eval_metrics.json"
print('백업 완료')

백업 완료


## 8. 추론 → submission.csv

In [ ]:
%cd /content
!python src/inference.py

/content
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
1. Preparing retriever from train data...
2. Loading LLM...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.0.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 434/434 [00:00<00:00, 590.68it/s, Materializing param=model.norm.weight]
unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Unsloth 2026.5.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
3. Running streaming inference...
Chunks: 18it [1:18:18, 261

## 9. 제출 파일 점검

> **[Claude 공유 필수 #2]**  
> 아래 셀을 실행한 뒤 **출력 결과 전체를 복사해서 Claude에게 붙여넣기**  
> (rows 수, op 분포, 빈 target_id, 결측값 확인)


In [ ]:
import pandas as pd
df = pd.read_csv('/content/submission.csv')
print('rows:', len(df))
print('\nop 분포:')
print(df['op'].value_counts(dropna=False))
print('\n빈 target_id:', (df['target_id'].isna() | (df['target_id'] == '')).sum())
print('CLICK에 value 있음:', ((df['op'] == 'CLICK') & (df['value'].astype(str).str.strip() != '')).sum())
print('결측값:', df.isnull().sum().sum())
df.head(10)

rows: 4417

op 분포:
op
CLICK     2492
TYPE      1095
SELECT     830
Name: count, dtype: int64

빈 target_id: 0
CLICK에 value 있음: 2492
결측값: 2566


,id,op,target_id,value
0,aac_mix_test_000000,TYPE,elem_ea6fee3c,2026-05-08
1,aac_mix_test_000001,CLICK,elem_b21884bd,NaN
2,aac_mix_test_000002,CLICK,elem_a14334ec,NaN
3,aac_mix_test_000003,CLICK,elem_b8db87e8,NaN
4,aac_mix_test_000004,CLICK,elem_4ae73ff2,NaN
5,aac_mix_test_000005,TYPE,elem_839b03d3,2026-12-23
6,aac_mix_test_000006,CLICK,elem_1a41eee7,NaN
7,aac_mix_test_000007,SELECT,elem_36619baf,Production
8,aac_mix_test_000008,SELECT,elem_0cfda850,NaN
9,aac_mix_test_000009,CLICK,elem_a95a67c5,NaN


## 10. submission.csv Drive 저장

In [7]:
!cp /content/submission.csv "$DRIVE_ROOT/submission.csv"
print('Drive 저장 완료')

cp: cannot stat '/content/submission.csv': No such file or directory
Drive 저장 완료


In [6]:
from google.colab import files
files.download('/content/submission.csv')

FileNotFoundError: Cannot find file: /content/submission.csv